In [0]:
import requests
import json
from datetime import datetime, timezone

FIRE_HISTORY_API_URL = (
    "https://services.arcgis.com/V6ZHFr6zdgNZuVG0/"
    "ArcGIS/rest/services/"
    "California_Fire_Perimeters/"
    "FeatureServer/0/query"
)

BRONZE_FIRE_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "bronze/fire_history"
)

MIN_LONGITUDE = -118.95
MIN_LATITUDE = 33.95
MAX_LONGITUDE = -118.45
MAX_LATITUDE = 34.25

print("Fire history configuration loaded.")

In [0]:
study_envelope = {
    "xmin": MIN_LONGITUDE,
    "ymin": MIN_LATITUDE,
    "xmax": MAX_LONGITUDE,
    "ymax": MAX_LATITUDE,
    "spatialReference": {"wkid": 4326}
}

params = {
    "where": "1=1",
    "geometry": json.dumps(study_envelope),
    "geometryType": "esriGeometryEnvelope",
    "spatialRel": "esriSpatialRelIntersects",
    "inSR": "4326",
    "outSR": "4326",
    "outFields": (
        "OBJECTID,YEAR_,FIRE_NAME,ALARM_DATE,"
        "CONT_DATE,CAUSE,GIS_ACRES,AGENCY,DECADES"
    ),
    "returnGeometry": "true",
    "f": "geojson"
}

response = requests.get(
    FIRE_HISTORY_API_URL,
    params=params,
    timeout=120
)

response.raise_for_status()

fire_data = response.json()

print("Fire history API request successful")
print("Features returned:", len(fire_data["features"]))

In [0]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

output_path = f"{BRONZE_FIRE_PATH}/load_timestamp={run_timestamp}"

fire_json = json.dumps(fire_data)

raw_fire_df = spark.createDataFrame(
    [(fire_json, run_timestamp, len(fire_data["features"]))],
    ["raw_json", "ingestion_timestamp", "feature_count"]
)

raw_fire_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

print("Bronze fire history ingestion successful")
print("Features stored:", len(fire_data["features"]))
print("Path:", output_path)

In [0]:
check_fire = spark.read.format("delta").load(output_path)

display(
    check_fire.select(
        "ingestion_timestamp",
        "feature_count"
    )
)